#### From Text to Diagnosis:
#### Evaluating GPT-Based Models for Classifying Depression Severity in Online Texts

# Script 2.2 - PROMPT ENGINEERING (Batch)

Adapted from non-Batch version with GPT 4 assistance for ease of implementation.

In [ ]:
### IMPORTS ###
!pip install --upgrade openai
import os
import time
import requests
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, confusion_matrix, precision_score, recall_score, f1_score, classification_report

In [ ]:
api_key = input('Enter your OpenAI API key: ')
os.environ['OPENAI_API_KEY'] = api_key
headers = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json'
}

In [ ]:
# load Dataset.
df = pd.read_csv(r"C:\Users\marti\Downloads\RAG_test_set (1).csv", delimiter=',')
# Param. to impose row limit to reduce spending; used ONLY for testing.
limit_rows = False
if limit_rows:
    df = df.head(10)

# Prompting 

In [ ]:
# Define the function to construct prompts
def construct_prompt(text, prompt_type, examples=None):
    examples_str = ""
    if examples:
        # Simply join the examples with newlines since each is a complete string
        examples_str = "\n\n".join(examples)

# Baseline Zero-Shot
    if prompt_type == 'baseline':
        return f"""
Question: What is the depression level of this user?
Output Constraint: Only return the classification of depression level (Minimum, Mild, Moderate, Severe).
Classify the following text: 
```{text}```
Format the output in JSON format. For example:
{{
    "depression_level": "Mild"
}}
"""

# Zero-Shot
    elif prompt_type == 'zero-shot':
        return f"""
Instruction: You will be given text data from a user's Reddit posts. \
Assume the role of a psychologist and classify the depression level of the user \
based on the text from the patient's Reddit posts. The depression level is \
on a scale of Minimum to Severe, where Minimum is no to minimal signs of depression, \
Mild is mild signs of depression, Moderate is moderate signs of depression, \
and Severe is very depressed.

Classify the following text: 
```{text}```
Format the output in JSON format. For example:
{{
    "depression_level": "Mild"
}}
"""    

# Few-shot Context and Role
    elif prompt_type == 'few-shot':
        return f"""
Instruction: You will be given text data from a user's Reddit posts. \
Assume the role of a psychologist and classify the depression level of the user \
based on the text from the patient's Reddit posts. The depression level is \
on a scale of Minimum to Severe, where Minimum is no to minimal signs of depression, \
Mild is mild signs of depression, Moderate is moderate signs of depression, \
and Severe is very depressed.

Here are some examples for context (delimited by double quotes):

"[Wtf. I did this to myself. I knew I needed to study and I just kept putting it off \
and putting it off. I knew I didnâ€™t do well but I didnâ€™t think I bombed it so fucking hard. \
Iâ€™m going to have to lie to everyone about it.]" ["Minimum"]

"[I don't know if I'm able to do that, since I didn't tell the doctor the truth to begin with. \
I will say that my doctor very clearly didn't believe me at my follow up appointment, but my \
bf was in the waiting room. So I didn't admit to anything. I was teary eyed, however. \
From anyone's experience- is it worth it to go to the police?]" ["Mild"]

"[I drove the 10 minutes home with my fiance on the phone helping me count my breathing. \
I got home and collapsed. I couldn't breathe. My neck and chest hurt. I was hysterical.]" ["Moderate"]

"[It feels great, but I'm worried. I tried to kill myself one year and one month ago. \
I am prone to depression. I am worried about crashing too hard, and I have kids to care for. \
Has anyone else experienced anything like this?]" ["Severe"]

Classify the following text: 
```{text}```
Format the output in JSON format. For example:
{{
    "depression_level": "Mild"
}}
"""

# Zero-Shot Chain of Thought 
    elif prompt_type == 'zero-shot chain of thought':
        return f"""
Instruction: You are a psychologist analyzing text data from a user's reddit posts \
to determine their depression level. Classify the depression level of the user based \
on the text provided. The depression level is on a scale of Minimum to Severe, where: \
Minimum = No to minimal signs of depression
Mild = Mild signs of depression
Moderate = Moderate signs of depression
Severe = Severe signs of depression
Use the following chain of thought to analyze the text:
1. Identify any explicit statements related to feelings of sadness, hopelessness, failure, or unhappiness.
2. Assess the frequency and intensity of these feelings based on the user's descriptions.
3. Consider any additional context that might indicate a higher or lower depression level.
4. Based on your assessment, assign a depression level from Minimum to Severe.
Question: What is the depression level of this user?
Output Constraint: Only return the classification of depression level (Minimum, Mild, Moderate, Severe) \
in a JSON format.

Classify the following text: 
```{text}```

Example Output:
{{
    "depression_level": "Severe"
}}
"""

# Few-Shot Chain of thought 
    elif prompt_type == 'few-shot chain of thought':
        return f"""
Instruction: You are a psychologist analyzing text data from a user's reddit posts \ 
to determine their depression level.\
Classify the depression level of the user based on the text provided. \
The depression level is on a scale of Minimum to Severe, where: 
Minimum = No to minimal signs of depression
Mild = Mild signs of depression
Moderate = Moderate signs of depression
Severe = Severe signs of depression
Use the following chain of thought to analyze the text:
1. Identify any explicit statements related to feelings of \
sadness, hopelessness, failure, or unhappiness.
2. Assess the frequency and intensity of these feelings based on the user's descriptions.
3. Consider any additional context that might indicate a higher or lower depression level.
4. Based on your assessment, assign a depression level from Minimum to Severe.
These are examples of your previous work:

"[Wtf. I did this to myself. I knew I needed to study and I just kept putting it off \
and putting it off. I knew I didnâ€™t do well but I didnâ€™t think I bombed it so fucking hard. \
Iâ€™m going to have to lie to everyone about it.]" ["Minimum"]

"[I don't know if I'm able to do that, since I didn't tell the doctor the truth to begin with. \
I will say that my doctor very clearly didn't believe me at my follow up appointment, but my \
bf was in the waiting room. So I didn't admit to anything. I was teary eyed, however. \
From anyone's experience- is it worth it to go to the police?]" ["Mild"]

"[I drove the 10 minutes home with my fiance on the phone helping me count my breathing. \
I got home and collapsed. I couldn't breathe. My neck and chest hurt. I was hysterical.]" ["Moderate"]

"[It feels great, but I'm worried. I tried to kill myself one year and one month ago. \
I am prone to depression. I am worried about crashing too hard, and I have kids to care for. \
Has anyone else experienced anything like this?]" ["Severe"]

Question: What is the depression level of this user?
Output Constraint: Only return the classification of depression level (Minimum, Mild, Moderate, Severe) \
in a JSON format.
Classify the following text: 
```{text}```

Example Output:
{{
    "depression_level": "Severe"
}}
"""

# RAG with one example per class
    elif prompt_type == 'RAG One':
        return f"""
Instruction: You will be given text data from a user. \
Assume the role of a psychologist and classify the depression level\
of the user based on the text from the patient's Reddit posts. \
The depression level is on a scale of Minimum to Severe, \
where Minimum is no to minimal signs of depression, \
Mild is mild signs of depression, \
Moderate is moderate signs of depression, \
and Severe is very depressed.\
Here are some examples for context (delimited by double quotes):
{examples_str}
Classify the following text: 
```{text}```
Format the output in JSON format. For example:
{{
    "depression_level": "Mild"
}}
"""         
# RAG with two examples per class    
    elif prompt_type == 'RAG Two':
        return f"""
Instruction: You will be given text data from a user. \
Assume the role of a psychologist and classify the depression level\
of the user based on the text from the patient's Reddit posts. \
The depression level is on a scale of Minimum to Severe, \
where Minimum is no to minimal signs of depression, \
Mild is mild signs of depression, \
Moderate is moderate signs of depression, \
and Severe is very depressed.\
Here are some examples for context (delimited by double quotes):
{examples_str}
Classify the following text: 
```{text}```
Format the output in JSON format. For example:
{{
    "depression_level": "Mild"
}}
"""         
    else:
        return f"Unrecognized prompt type: {prompt_type}"

In [ ]:
# Define the function to extract RAG examples from a row
def get_rag_examples_from_row(row, example_count):
    examples = []
    for severity in ['minimum', 'mild', 'moderate', 'severe']:
        for i in range(1, example_count + 1):  # Dynamic number of examples per severity
            example_text = row.get(f"{severity}{i}")
            if pd.notna(example_text):
                examples.append(example_text)
    return examples

# Create and Download Batches

In [ ]:
# Define the function to create and send requests
def create_and_send_requests(df, prompt_type, gpt_model, example_count):
    url = "https://api.openai.com/v1/chat/completions"

    for i in range(len(df)):
        text = df.at[i, 'text']
        examples = get_rag_examples_from_row(df.iloc[i], example_count)  # Use the new parameter here
        prompt = construct_prompt(text, prompt_type, examples)
        request_body = {
            "model": gpt_model,
            "messages": [{"role": "user", "content": prompt.strip()}],
            "temperature": 0.2
        }

        response = requests.post(url, headers=headers, data=json.dumps(request_body))
        
        # Print the response for debugging purposes
        print(response.json())

# Define the function to create JSONL files for batch processing
def create_jsonl_file(df, prompt_type, filename, gpt_model):
    with open(filename, 'w') as f:
        for i in range(len(df)):
            text = df.at[i, 'text']
            examples = get_rag_examples_from_row(df.iloc[i], 2)  # Extract RAG examples from the current row
            prompt = construct_prompt(text, prompt_type, examples)
            request = {
                "custom_id": f"request-{i}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": gpt_model,
                    "messages": [{"role": "user", "content": prompt}],
                    "temperature": 0.2
                }
            }
            f.write(json.dumps(request) + "\n")

# Define the function to upload the file to OpenAI
def upload_file_to_openai(filename):
    with open(filename, 'rb') as f:
        response = requests.post(
            'https://api.openai.com/v1/files',
            headers={
                'Authorization': f'Bearer {api_key}'
            },
            files={
                'file': (filename, f),
                'purpose': (None, 'batch')
            }
        )
    response.raise_for_status()  # Raise an error for bad responses
    return response.json()['id']

# Define the function to create a batch job
def create_batch_job(input_file_id, endpoint='/v1/chat/completions', completion_window='24h'):
    response = requests.post(
        'https://api.openai.com/v1/batches',
        headers=headers,
        json={
            "input_file_id": input_file_id,
            "endpoint": endpoint,
            "completion_window": completion_window
        }
    )
    response.raise_for_status()  # Raise an error for bad responses
    return response.json()

# Define the function to retrieve batch results
def retrieve_batch_results(batch_job_id):
    response = requests.get(
        f'https://api.openai.com/v1/batches/{batch_job_id}',
        headers=headers
    )
    response.raise_for_status()  # Raise an error for bad responses
    return response.json()

# Define the function to download the result file
def download_file(file_id):
    response = requests.get(
        f'https://api.openai.com/v1/files/{file_id}/content',
        headers=headers
    )
    response.raise_for_status()  # Raise an error for bad responses
    with open(f'{file_id}.jsonl', 'wb') as f:
        f.write(response.content)
    return f'{file_id}.jsonl'

# Process Results and Calculate Metrics


In [ ]:
# Define the function to normalize key names
def normalize_key_names(response_json):
    normalized_response = {}
    for key in response_json:
        normalized_key = key.lower().replace(" ", "_")
        normalized_response[normalized_key] = response_json[key]
    return normalized_response

In [ ]:
# Define the function to calculate and print metrics
def calculate_and_print_metrics(results_df, prompt_type):
    if not results_df.empty and 'match' in results_df.columns:
        results_df = results_df.drop(columns=['index'])
        results_df = results_df.dropna(subset=['predicted_depression_level', 'actual_depression_level'])
        results_df = results_df[results_df['predicted_depression_level'].apply(lambda x: isinstance(x, int))]
        results_df['predicted_depression_level'] = results_df['predicted_depression_level'].astype(int)
        results_df['actual_depression_level'] = results_df['actual_depression_level'].astype(int)

        accuracy = accuracy_score(results_df['actual_depression_level'], results_df['predicted_depression_level'])
        mae = mean_absolute_error(results_df['actual_depression_level'], results_df['predicted_depression_level'])
        mse = mean_squared_error(results_df['actual_depression_level'], results_df['predicted_depression_level'])
        rmse = np.sqrt(mse)
        conf_matrix = confusion_matrix(results_df['actual_depression_level'], results_df['predicted_depression_level'])
        precision = precision_score(results_df['actual_depression_level'], results_df['predicted_depression_level'], average='macro')
        recall = recall_score(results_df['actual_depression_level'], results_df['predicted_depression_level'], average='macro')
        f1 = f1_score(results_df['actual_depression_level'], results_df['predicted_depression_level'], average='macro')

        print(f"Results for prompt type: {prompt_type}")
        print(f"Accuracy: {accuracy * 100:.2f}%")
        print(f"Mean Absolute Error (MAE): {mae}")
        print(f"Mean Squared Error (MSE): {mse}")
        print(f"Root Mean Squared Error (RMSE): {rmse}")
        print("Confusion Matrix:")
        print(conf_matrix)
        print(f"Precision Score: {precision}")
        print(f"Recall Score: {recall}")
        print(f"F1 Score: {f1}")

        class_report = classification_report(
            results_df['actual_depression_level'],
            results_df['predicted_depression_level'],
            target_names=['Class 0', 'Class 1', 'Class 2', 'Class 3']
        )
        print("Classification Report:")
        print(class_report)
    else:
        print(f"No valid results to display for prompt type: {prompt_type} or 'match' column missing.")


In [ ]:
# Define the function to process batch results
def process_batch_results(results_filename, df, prompt_type):
    label_mapping = {"minimum": 0, "mild": 1, "moderate": 2, "severe": 3, 0: 0, 1: 1, 2: 2, 3: 3}
    # Load and process the results
    results = []
    with open(results_filename, 'r') as f:
        for line in f:
            result = json.loads(line)
            response = result['response']['body']['choices'][0]['message']['content']
            response = response.strip().strip('```').strip('json').strip()
            
            # Debugging: Print the response content
            print(f"Response Content: {response}")
            
            try:
                response_json = json.loads(response)
            except json.JSONDecodeError as e:
                print(f"JSONDecodeError: {e}")
                continue
            
            response_json = normalize_key_names(response_json)
            depression_level = response_json.get('depression_level', None)
            if depression_level is None:
                continue
            if isinstance(depression_level, str):
                depression_level = label_mapping.get(depression_level.lower(), -1)
            elif isinstance(depression_level, int):
                depression_level = label_mapping.get(depression_level, -1)
            if depression_level == -1:
                continue
            actual_label = int(df.at[int(result['custom_id'].split('-')[1]), 'multi_label_number'])
            result = {
                "index": result['custom_id'],
                "predicted_depression_level": depression_level,
                "actual_depression_level": actual_label,
                "match": depression_level == actual_label
            }
            results.append(result)
    results_df = pd.DataFrame(results)
    calculate_and_print_metrics(results_df, prompt_type)

# Main Loop to Create Batch Jobs, Download Them and Process Results

In [ ]:
gpt_models = [#'gpt-3.5-turbo', 
    'gpt-4o'
             ]
prompt_types = [
    'baseline', 'zero-shot', 'few-shot','zero-shot chain of thought', 'few-shot chain of thought','RAG Two', #'RAG One',   
] 
example_count = 2  # Set to 1 or 2 based on the desired number of examples

batch_jobs = []

for gpt_model in gpt_models:
    for prompt in prompt_types:
        jsonl_filename = f'batch_requests_{prompt.replace(" ", "_")}.jsonl'
        create_jsonl_file(df, prompt, jsonl_filename, gpt_model)
        input_file_id = upload_file_to_openai(jsonl_filename)
        batch_job = create_batch_job(input_file_id)
        batch_jobs.append((batch_job['id'], prompt))
        print(f"Batch job created with ID: {batch_job['id']} for prompt: {prompt}")

In [ ]:
# Wait for all batch jobs to complete and download results
for batch_job_id, prompt in batch_jobs:
    batch_results = retrieve_batch_results(batch_job_id)
    while batch_results['status'] not in ['completed', 'failed']:
        time.sleep(60)  # Wait for a minute before checking the status again
        batch_results = retrieve_batch_results(batch_job_id)
    
    # Debugging information
    print(f"Batch results: {batch_results}")
    
    if batch_results['status'] == 'completed':
        output_file_id = batch_results.get('output_file_id', None)
        
        # Additional check to ensure output_file_id is not None
        if output_file_id:
            result_file = download_file(output_file_id)
            print(f"Results downloaded to: {result_file}")
            # Process and print the results
            process_batch_results(result_file, df, prompt)
        else:
            print(f"Output file ID is None for batch job: {batch_job_id}")
    else:
        print(f"Batch processing failed for prompt: {prompt}")